# 01. Data Ingestion, Quality Audit & Exploratory Data Analysis (EDA)
**Author**: Member 1 (Data Analyst)  
**Project**: Smart Loan Default & Credit Risk Assessment System  
**Module**: Machine Learning Module - Group Project Assignment

## 1. Environment Setup & Data Loading
Import required analytical libraries and load the raw Credit Risk dataset.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11

raw_path = os.path.join('..', 'data', 'raw', 'credit_risk_dataset.csv')
df = pd.read_csv(raw_path)
print(f'Raw dataset shape: {df.shape[0]} rows, {df.shape[1]} columns.')
df.head()

## 2. Dataset Schema & Quality Audit
Inspect data types, missing values, duplicates, and initial summary statistics.

In [ ]:
print('=== Data Types ===')
print(df.dtypes)

print('\n=== Missing Values Summary ===')
null_summary = pd.DataFrame({
    'Missing Count': df.isnull().sum(),
    'Percentage (%)': (df.isnull().sum() / len(df) * 100).round(2)
})
print(null_summary[null_summary['Missing Count'] > 0])

print(f'\nExact duplicate rows count: {df.duplicated().sum()}')

## 3. Data Cleaning & Anomaly Resolution
1. Drop duplicate rows.
2. Filter biologically impossible ages (age > 85) and invalid employment lengths.
3. Impute `loan_int_rate` using grouped median by `loan_grade`.
4. Impute `person_emp_length` using grouped median by age brackets.

In [ ]:
# 1. Drop duplicates
df_clean = df.drop_duplicates().copy()

# 2. Filter biological/physical anomalies
df_clean = df_clean[(df_clean['person_age'] >= 18) & (df_clean['person_age'] <= 85)]
df_clean = df_clean[df_clean['person_emp_length'] <= (df_clean['person_age'] - 16)]

# 3. Grouped median imputation: loan_int_rate by loan_grade
grade_medians = df_clean.groupby('loan_grade')['loan_int_rate'].transform('median')
df_clean['loan_int_rate'] = df_clean['loan_int_rate'].fillna(grade_medians)

# 4. Grouped median imputation: person_emp_length by age group
age_bins = pd.cut(df_clean['person_age'], bins=[17, 25, 35, 50, 100], labels=['18-25', '26-35', '36-50', '50+'])
emp_medians = df_clean.groupby(age_bins, observed=False)['person_emp_length'].transform('median')
df_clean['person_emp_length'] = df_clean['person_emp_length'].fillna(emp_medians)

print(f'Cleaned dataset shape: {df_clean.shape}')
print(f'Remaining missing values across all columns: {df_clean.isnull().sum().sum()}')

## 4. Target Variable Analysis (`loan_status`)
Examining the class imbalance between Non-Default (0) and Default (1).

In [ ]:
counts = df_clean['loan_status'].value_counts()
proportions = df_clean['loan_status'].value_counts(normalize=True) * 100

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(['Non-Default (0)', 'Default (1)'], counts, color=['#10b981', '#ef4444'], width=0.45)
for bar, prop in zip(bars, proportions):
    yval = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, yval + 400, f'{prop:.1f}% ({int(yval):,})', ha='center', fontweight='bold')
ax.set_ylabel('Applicant Count')
ax.set_title('Target Class Distribution (loan_status)', fontweight='bold', pad=12)
plt.tight_layout()
plt.show()

## 5. Export Cleaned Dataset for Member 2
Export the verified dataset to `ml/data/processed/cleaned_credit_data.csv`.

In [ ]:
processed_path = os.path.join('..', 'data', 'processed', 'cleaned_credit_data.csv')
os.makedirs(os.path.dirname(processed_path), exist_ok=True)
df_clean.to_csv(processed_path, index=False)
print(f'Successfully saved clean dataset to {processed_path}')